## 1. Importações e Configuração

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120


### MLflow — Configuração

In [54]:
import mlflow
import mlflow.sklearn
import subprocess, time, os, requests
import warnings
warnings.filterwarnings("ignore")

MLFLOW_PORT    = 5000
MLFLOW_DIR     = os.path.expanduser("./mlflow-local")
DB_PATH        = os.path.join(MLFLOW_DIR, "mlflow.db")
ARTIFACT_DIR   = os.path.join(MLFLOW_DIR, "artifacts")
EXPERIMENT_NAME = "narwhal"

os.makedirs(ARTIFACT_DIR, exist_ok=True)

def server_running():
    try:
        return requests.get(f"http://localhost:{MLFLOW_PORT}/health", timeout=2).status_code == 200
    except Exception:
        return False

if server_running():
    print(f"✅ Servidor MLflow já rodando → http://localhost:{MLFLOW_PORT}")
else:
    log = open(os.path.join(MLFLOW_DIR, "server.log"), "w")
    proc = subprocess.Popen(
        ["mlflow", "server",
         "--host", "127.0.0.1", "--port", str(MLFLOW_PORT),
         "--backend-store-uri", f"sqlite:///{DB_PATH}",
         "--default-artifact-root", ARTIFACT_DIR],
        stdout=log, stderr=log
    )
    print(f"Iniciando servidor MLflow (PID {proc.pid})...")
    for _ in range(20):
        time.sleep(1)
        if server_running():
            break
    if server_running():
        print(f"Servidor MLflow rodando → http://localhost:{MLFLOW_PORT}")
    else:
        print("Falha ao iniciar. Veja ~/mlflow-local/server.log")

# ── Conecta e cria/reutiliza o experimento ─────────────────────────────────────
mlflow.set_tracking_uri(f"http://localhost:{MLFLOW_PORT}")
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Experimento: '{EXPERIMENT_NAME}'")


✅ Servidor MLflow já rodando → http://localhost:5000
Experimento: 'narwhal'


# 2. Exemplos

## Para pegar o id do experimento

In [41]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

EXPERIMENT_NAME = "narwhal"
experiment_id = client.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
print(experiment_id)

2


## Listar todas as runs de um experimento

In [71]:
df = mlflow.search_runs(
    experiment_ids=[experiment_id],
)

df.head(10)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.roc_auc,metrics.system/mem_pico_treino_mb,metrics.system/tempo_por_amostra_ms,metrics.system/tempo_inferencia_s,...,tags.mlflow.source.name,tags.mlflow.user,tags.mlflow.loggedArtifacts,tags.split,tags.mlflow.runName,tags.framework,tags.cpu_nucleos,tags.mlflow.source.type,tags.so,tags.dataset
0,1061fc46431b4203ac151a03cb7d3aba,2,FINISHED,file:c:/Users/vitor/Documents/narwal-tcc/data-...,2026-06-17 22:34:49.128000+00:00,2026-06-17 22:34:53.452000+00:00,0.7992,0.0031,0.0032,0.000166,...,modelo_arvore_mlflow.ipynb,vitor,"[{""path"": ""matriz_confusao.json"", ""type"": ""tab...",75/25,dt-depth7-gini,scikit-learn,12,NOTEBOOK,Windows 11,sonar_data_matriz.csv
1,ffef50c6e63f4756a9db6086106df153,2,FINISHED,file:c:/Users/vitor/Documents/narwal-tcc/data-...,2026-06-17 22:33:07.154000+00:00,2026-06-17 22:33:11.478000+00:00,0.7992,0.0031,0.0028,0.000146,...,modelo_arvore_mlflow.ipynb,vitor,"[{""path"": ""matriz_confusao.json"", ""type"": ""tab...",75/25,dt-depth5-gini,scikit-learn,12,NOTEBOOK,Windows 11,sonar_data_matriz.csv
2,943e8f2434f84b85b4ecb5e7d3a9a578,2,FINISHED,file:c:/Users/vitor/Documents/narwal-tcc/data-...,2026-06-17 22:32:48.285000+00:00,2026-06-17 22:32:53.044000+00:00,0.8500,0.0031,0.0057,0.000297,...,modelo_arvore_mlflow.ipynb,vitor,"[{""path"": ""matriz_confusao.json"", ""type"": ""tab...",75/25,dt-depth7-entropy,scikit-learn,12,NOTEBOOK,Windows 11,sonar_data_matriz.csv
3,549599ca1dcc4bbe9b335e891c6c35c8,2,FINISHED,file:c:/Users/vitor/Documents/narwal-tcc/data-...,2026-06-17 22:31:59.766000+00:00,2026-06-17 22:32:04.303000+00:00,0.8280,0.0031,0.0030,0.000154,...,modelo_arvore_mlflow.ipynb,vitor,"[{""path"": ""matriz_confusao.json"", ""type"": ""tab...",75/25,dt-depth5-entropy,scikit-learn,12,NOTEBOOK,Windows 11,sonar_data_matriz.csv


## Carregar uma run específica pelo run_id

In [72]:
run = mlflow.get_run("549599ca1dcc4bbe9b335e891c6c35c8")
params = run.data.params
metrics = run.data.metrics
tags = run.data.tags

print(metrics)
print(params)
print(tags)

{'accuracy': 0.7885, 'precision': 0.913, 'recall': 0.7, 'f1_score': 0.7925, 'roc_auc': 0.828, 'cv_accuracy_mean': 0.6108, 'cv_accuracy_std': 0.075, 'system/tempo_treino_s': 0.005121, 'system/tempo_inferencia_s': 0.000154, 'system/tempo_por_amostra_ms': 0.003, 'system/mem_pico_treino_mb': 0.0031, 'system/delta_ram_mb': 0.19, 'system/delta_cpu_percent': 5.2}
{'criterion': 'entropy', 'max_depth': '5', 'min_samples_leaf': '4', 'random_state': '42'}
{'mlflow.user': 'vitor', 'mlflow.source.name': 'modelo_arvore_mlflow.ipynb', 'mlflow.source.type': 'NOTEBOOK', 'mlflow.runName': 'dt-depth5-entropy', 'dataset': 'sonar_data_matriz.csv', 'framework': 'scikit-learn', 'split': '75/25', 'so': 'Windows 11', 'cpu_nucleos': '12', 'mlflow.loggedArtifacts': '[{"path": "matriz_confusao.json", "type": "table"}]'}


## Registrar modelo

In [79]:
from mlflow import MlflowClient

client = MlflowClient()

best_run_id = "549599ca1dcc4bbe9b335e891c6c35c8"

best_logged_model = next(
    m for m in client.search_logged_models(experiment_ids=["2"])
    if m.source_run_id == best_run_id
)

mlflow.register_model(
    model_uri=best_logged_model.model_uri,
    name="DecisionTreeClassifier"
)

Successfully registered model 'DecisionTreeClassifier'.
2026/06/17 19:58:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: DecisionTreeClassifier, version 1
Created version '1' of model 'DecisionTreeClassifier'.


<ModelVersion: aliases=[], creation_timestamp=1781737109617, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1781737109617, metrics=None, model_id=None, name='DecisionTreeClassifier', params=None, run_id='549599ca1dcc4bbe9b335e891c6c35c8', run_link='', source='models:/m-d88ba4d9a69141708485915e6fd37892', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

## Promover o modelo utilizando alias

In [82]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

client.set_registered_model_alias(
    name="DecisionTreeClassifier",
    alias="champion",
    version=1
)

## Utilizando o modelo

In [86]:
## Atenção: Testando o modelo com os dados de treinamento

model = mlflow.pyfunc.load_model(
    "models:/DecisionTreeClassifier@champion"
)

PATH = r"../sonar_data_matriz.csv"

dataset = pd.read_csv(PATH, header=None)

X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

# Codificação: M=1 (Mina), R=0 (Rocha)
y = np.where(y == 'M', 1, 0)

X_test = X[100:,:]
Y_test = y[100:]

preds = model.predict(X_test)

comp = pd.DataFrame({"Real": Y_test, "Predito": preds})
percentual_acertos = (comp["Real"] == comp["Predito"]).mean() * 100

print(f"Acurácia no teste: {percentual_acertos:.2f}%")

Acurácia no teste: 84.26%
